# 22 MCP、Skills 与 SOP（MCP, Skills & SOPs） / MCP, Skills & SOPs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/22-mcp-skills-sop/22-mcp-skills-sop.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/22-mcp-skills-sop/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/22-mcp-skills-sop/interactive/mcp.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/22-mcp-skills-sop)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "22-mcp-skills-sop"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)
try:
    import langgraph  # noqa: F401
except ImportError:  # e.g. in Colab / 例如在 Colab 中
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.19"], check=True)
try:
    import mcp  # noqa: F401
except ImportError:  # e.g. in Colab / 例如在 Colab 中
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0"], check=True)

## `mcp_from_scratch.py`

```text
The Model Context Protocol from scratch / 从零实现 MCP.

Run / 运行:
    python docs/22-mcp-skills-sop/code/mcp_from_scratch.py

Covers / 包含:
    1. JSON-RPC 2.0 messages over stdio: requests, responses, errors
       stdio 上的 JSON-RPC 2.0 消息：请求、响应、错误
    2. a server with the three server features: tools, resources and prompts, plus server/discover
       一个服务器，提供三种能力：工具、资源和提示，以及 server/discover
    3. a client that starts the server as a subprocess and stamps every request with _meta
       (protocol revision 2026-07-28 is stateless: no initialize handshake)
       一个客户端：把服务器作为子进程启动，每个请求都带上 _meta（2026-07-28 版协议是无状态的，没有 initialize 握手）
    4. a host: chapter 21's agent loop with tools discovered from the server; it gives the same
       answers as with local tools
       宿主：21 章的 agent loop，工具从服务器发现而来；答案与使用本地工具时完全相同
    5. several servers at once: name clashes, and how tool descriptions pile up in every prompt
       同时连接多个服务器：重名，以及工具说明怎样在每次调用的提示里越堆越多

Only the standard library; no API key. The server here is the chapter 21 world of documents.
只用标准库，不需要 API key。服务器提供的是 21 章那个文档世界。
```

In [ ]:
__file__ = str(CODE / "mcp_from_scratch.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import json
import re
import subprocess
import sys
from pathlib import Path

In [ ]:
HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent.parent / "21-agents" / "code"))

In [ ]:
from agent_loop import TOOLS as LOCAL_TOOLS  # noqa: E402
from agent_loop import ScriptedModel, agent_loop, make_tasks, make_world, run_tool  # noqa: E402

In [ ]:
PROTOCOL = "2026-07-28"
META_KEY = "io.modelcontextprotocol/"
# JSON-RPC error codes / JSON-RPC 错误码
PARSE_ERROR, INVALID_REQUEST, METHOD_NOT_FOUND, INVALID_PARAMS, UNSUPPORTED_VERSION = -32700, -32600, -32601, -32602, -32022

### 2. The server / 服务器

In [ ]:
class DocsServer:
    """An MCP server for the documents of chapter 21. It only maps JSON-RPC requests to Python
    functions; the transport (stdio) is in serve_stdio below.
    21 章文档的 MCP 服务器。它只负责把 JSON-RPC 请求映射到 Python 函数；传输（stdio）在下面的 serve_stdio 里。"""

    name, version = "docs", "1.0"

    def __init__(self, world):
        self.world = world
        self.tools = {  # name -> (description, input schema, annotations, function)
            "search": ("Search the documents. Returns the titles of the best matches.",
                       {"query": "string"}, {"readOnlyHint": True}, lambda a: run_tool(world, "search", a)),
            "read": ("Read the document with this exact title.",
                     {"title": "string"}, {"readOnlyHint": True}, lambda a: run_tool(world, "read", a)),
            "calculator": ("Evaluate an arithmetic expression such as 2026 - 1950.",
                           {"expression": "string"}, {"readOnlyHint": True}, lambda a: calculate(world, a["expression"])),
        }
        self.methods = {
            "server/discover": self.discover,
            "tools/list": self.list_tools, "tools/call": self.call_tool,
            "resources/list": self.list_resources, "resources/read": self.read_resource,
            "prompts/list": self.list_prompts, "prompts/get": self.get_prompt,
        }

    # -- the protocol layer / 协议层 --
    def handle(self, msg):
        """One JSON-RPC message in, one response out (None for a notification). 一条消息进，一条响应出（通知没有响应）。"""
        if not isinstance(msg, dict) or msg.get("jsonrpc") != "2.0" or "method" not in msg:
            return error(msg.get("id") if isinstance(msg, dict) else None, INVALID_REQUEST, "Invalid Request")
        if "id" not in msg:
            return None  # a notification: nothing to answer / 通知：不用回复
        params = msg.get("params") or {}
        meta = params.get("_meta") or {}
        version = meta.get(META_KEY + "protocolVersion")
        if version is None:
            return error(msg["id"], INVALID_PARAMS, "every request must carry _meta with a protocolVersion")
        if version != PROTOCOL:
            return error(msg["id"], UNSUPPORTED_VERSION, "Unsupported protocol version",
                         {"requested": version, "supported": [PROTOCOL]})
        if msg["method"] not in self.methods:
            return error(msg["id"], METHOD_NOT_FOUND, f"Method not found: {msg['method']}")
        try:
            result = self.methods[msg["method"]]({k: v for k, v in params.items() if k != "_meta"})
        except ParamsError as e:
            return error(msg["id"], INVALID_PARAMS, str(e))
        result = {"resultType": "complete", **result, "_meta": {META_KEY + "serverInfo": {"name": self.name, "version": self.version}}}
        return {"jsonrpc": "2.0", "id": msg["id"], "result": result}

    # -- the features / 具体能力 --
    def discover(self, params):
        return {"supportedVersions": [PROTOCOL], "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
                "instructions": "Documents about made-up people and companies. Search, then read by exact title.",
                "ttlMs": 3_600_000, "cacheScope": "public"}

    def list_tools(self, params):
        tools = [{"name": n, "description": d,
                  "inputSchema": {"type": "object", "properties": {k: {"type": t} for k, t in s.items()}, "required": list(s)},
                  "annotations": ann} for n, (d, s, ann, _) in self.tools.items()]
        return {"tools": tools, "ttlMs": 3_600_000, "cacheScope": "public"}

    def call_tool(self, params):
        name, args = params.get("name"), params.get("arguments", {})
        if name not in self.tools:
            raise ParamsError(f"Unknown tool: {name}")
        _, spec, _, fn = self.tools[name]
        bad = [k for k in spec if not isinstance(args, dict) or not isinstance(args.get(k), str)]
        if bad:  # a tool-level error goes back to the model as a result with isError / 工具层面的错误作为结果返回给模型
            return {"content": [{"type": "text", "text": f"Invalid arguments: {bad} must be strings"}], "isError": True}
        text = fn(args)
        return {"content": [{"type": "text", "text": text}], "isError": text.startswith(("Error", "No document"))}

    def list_resources(self, params):
        docs = sorted(self.world["docs"])
        start = int(params.get("cursor") or 0)
        page = [{"uri": f"doc://{quote(t)}", "name": t, "mimeType": "text/plain"} for t in docs[start:start + 50]]
        out = {"resources": page, "ttlMs": 60_000, "cacheScope": "public"}
        if start + 50 < len(docs):
            out["nextCursor"] = str(start + 50)  # pagination / 分页
        return out

    def read_resource(self, params):
        uri = params.get("uri", "")
        title = unquote(uri.removeprefix("doc://"))
        if not uri.startswith("doc://") or title not in self.world["docs"]:
            raise ParamsError(f"Resource not found: {uri}")
        return {"contents": [{"uri": uri, "mimeType": "text/plain", "text": self.world["docs"][title]}], "ttlMs": 60_000, "cacheScope": "public"}

    def list_prompts(self, params):
        return {"prompts": [{"name": "answer_with_sources", "description": "Answer a question from documents, citing them.",
                             "arguments": [{"name": "question", "required": True}]}], "ttlMs": 3_600_000, "cacheScope": "public"}

    def get_prompt(self, params):
        if params.get("name") != "answer_with_sources" or "question" not in (params.get("arguments") or {}):
            raise ParamsError("prompts/get needs name='answer_with_sources' and a question")
        q = params["arguments"]["question"]
        text = (f"Answer the question using only the documents. Search first, read the documents by exact title, "
                f"and cite each title you used in [brackets].\nQuestion: {q}")
        return {"description": "Answer with sources", "messages": [{"role": "user", "content": {"type": "text", "text": text}}]}

In [ ]:
def calculate(world, expression):
    """'2026 - 1950' -> '2026 - 1950 = 76', so the result says what was computed. 结果里带上算式。"""
    value = run_tool(world, "calculator", {"expression": expression})
    return value if value.startswith("Error") else f"{expression} = {value}"

In [ ]:
class ParamsError(Exception):
    pass

In [ ]:
def error(id_, code, message, data=None):
    err = {"code": code, "message": message} | ({"data": data} if data is not None else {})
    return {"jsonrpc": "2.0", "id": id_, "error": err}

In [ ]:
def quote(title):
    return title.replace(" ", "%20")

In [ ]:
def unquote(s):
    return s.replace("%20", " ")

In [ ]:
def serve_stdio(server):
    """The stdio transport: one JSON-RPC message per line on stdin, one per line on stdout.
    Logs go to stderr, never stdout. stdio 传输：stdin 上每行一条消息，stdout 上每行一条；日志写到 stderr，绝不能写到 stdout。"""
    for line in sys.stdin:
        if not line.strip():
            continue
        try:
            msg = json.loads(line)
        except json.JSONDecodeError:
            reply = error(None, PARSE_ERROR, "Parse error")
        else:
            reply = server.handle(msg)
        if reply is not None:
            sys.stdout.write(json.dumps(reply) + "\n")
            sys.stdout.flush()

### 3. The client / 客户端

In [ ]:
class StdioClient:
    """Starts a server command as a subprocess and talks JSON-RPC to it over stdin/stdout.
    把服务器命令作为子进程启动，通过 stdin/stdout 与它说 JSON-RPC。"""

    def __init__(self, command, name="client", trace=None):
        self.proc = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
        self.next_id = 0
        self.info = {"name": name, "version": "0.1"}
        self.trace = trace  # a list to record every message, for printing / 记录每条消息

    def request(self, method, params=None):
        self.next_id += 1
        meta = {META_KEY + "protocolVersion": PROTOCOL, META_KEY + "clientCapabilities": {}, META_KEY + "clientInfo": self.info}
        msg = {"jsonrpc": "2.0", "id": self.next_id, "method": method, "params": {**(params or {}), "_meta": meta}}
        self.proc.stdin.write(json.dumps(msg) + "\n")
        self.proc.stdin.flush()
        reply = json.loads(self.proc.stdout.readline())
        if self.trace is not None:
            self.trace += [("→", msg), ("←", reply)]
        assert reply.get("id") == msg["id"]  # one request at a time here; real clients match ids / 真实客户端按 id 配对
        if "error" in reply:
            raise RuntimeError(f"{reply['error']['code']}: {reply['error']['message']}")
        return reply["result"]

    def send_raw(self, msg):
        """Send a message exactly as given (for showing errors). 原样发送一条消息（用来演示错误）。"""
        self.proc.stdin.write(json.dumps(msg) + "\n")
        self.proc.stdin.flush()
        return json.loads(self.proc.stdout.readline())

    def discover(self):
        return self.request("server/discover")

    def list_tools(self):
        return self.request("tools/list")["tools"]

    def call_tool(self, name, arguments):
        result = self.request("tools/call", {"name": name, "arguments": arguments})
        return "\n".join(c["text"] for c in result["content"] if c["type"] == "text"), result.get("isError", False)

    def close(self):
        self.proc.stdin.close()
        self.proc.wait(timeout=10)

In [ ]:
def server_command(script=__file__, *args):
    return [sys.executable, str(script), "--serve", *args]

### 4. The host: chapter 21's loop with MCP tools / 宿主：用 MCP 工具的 21 章循环

In [ ]:
FINAL_ANSWER = next(t for t in LOCAL_TOOLS if t["name"] == "final_answer")  # the host's own tool / 宿主自己的工具

In [ ]:
def mcp_to_loop_tools(mcp_tools, prefix=""):
    """MCP tool descriptions -> the format the chapter 21 loop puts in the system prompt.
    MCP 的工具说明 -> 21 章循环放进系统提示的格式。"""
    return [{"name": prefix + t["name"], "description": t["description"], "parameters": t["inputSchema"]} for t in mcp_tools]

In [ ]:
def make_caller(clients):
    """Route a tool name to the server that has it. 把工具名路由到提供它的服务器。"""
    def call(name, args):
        client, tool = clients[name]
        text, _ = client.call_tool(tool, args)
        return text
    return call

In [ ]:
def words(tools):
    return len(json.dumps(tools).split())

In [ ]:
def main():
    world = make_world()
    print("== 1-3. a session, message by message / 逐条消息看一次会话 ==")
    trace = []
    client = StdioClient(server_command(), name="demo-host", trace=trace)
    info = client.discover()
    tools = client.list_tools()
    text, is_error = client.call_tool("read", {"title": "Tafis Labs"})
    for arrow, m in trace:
        print(f"  {arrow} {json.dumps(m)[:150]}")
    assert info["supportedVersions"] == [PROTOCOL] and [t["name"] for t in tools] == ["search", "read", "calculator"]
    assert text.startswith("Tafis Labs was founded") and not is_error

    print("\n  resources, prompts and errors / 资源、提示和错误：")
    res = client.request("resources/list")
    print(f"  resources/list -> {len(res['resources'])} of {len(world['docs'])} documents, nextCursor={res.get('nextCursor')!r}")
    first = res["resources"][0]["uri"]
    print(f"  resources/read {first} -> {client.request('resources/read', {'uri': first})['contents'][0]['text'][:70]}")
    prompt = client.request("prompts/get", {"name": "answer_with_sources", "arguments": {"question": "Where does X work?"}})
    print(f"  prompts/get -> {prompt['messages'][0]['content']['text'][:80]}...")
    print(f"  tools/call with a bad argument -> isError={client.call_tool('read', {'title': 42})[1]}  (the model sees it and retries)")
    for method, params, code in [("tools/delete_everything", {}, METHOD_NOT_FOUND), ("resources/read", {"uri": "doc://Nobody"}, INVALID_PARAMS)]:
        try:
            client.request(method, params)
        except RuntimeError as e:
            print(f"  {method} -> error {e}")
            assert str(e).startswith(str(code))
    reply = client.send_raw({"jsonrpc": "2.0", "id": 99, "method": "tools/list", "params": {}})
    print(f"  a request without _meta -> {reply['error']['message']}")
    assert reply["error"]["code"] == INVALID_PARAMS
    client.close()

    print("\n== 4. the agent loop with tools from the server / 用服务器提供的工具运行 agent loop ==")
    client = StdioClient(server_command(), name="agent-host")
    loop_tools = mcp_to_loop_tools(client.list_tools()) + [FINAL_ANSWER]
    call = make_caller({t["name"]: (client, t["name"]) for t in loop_tools if t["name"] != "final_answer"})
    print(f"  system prompt tools: {[t['name'] for t in loop_tools]} ({words(loop_tools)} words; local version {words(LOCAL_TOOLS)})")
    tasks = make_tasks(world, n=30)
    same = total = 0
    for e in [0.0, 0.2]:
        for k, ts in tasks.items():
            local, remote = ScriptedModel(world, e, seed=k), ScriptedModel(world, e, seed=k)
            for question, truth in ts:
                a = agent_loop(local, world, question)
                b = agent_loop(remote, world, question, tools=loop_tools, call_tool=call)
                same += a[0] == b[0] and [m["content"] for m in a[3][1:]] == [m["content"] for m in b[3][1:]]
                total += 1
    print(f"  {same}/{total} runs (error rates 0% and 20%): same answer and same transcript as with local tools")
    assert same == total
    client.close()

    print("\n== 5. several servers / 多个服务器 ==")
    clients = {"docs": StdioClient(server_command(), "host"), "archive": StdioClient(server_command(), "host")}
    listed = {s: c.list_tools() for s, c in clients.items()}
    names = [t["name"] for ts in listed.values() for t in ts]
    clashes = sorted({n for n in names if names.count(n) > 1})
    print(f"  two servers both offer {clashes}: the host must namespace them, e.g. docs__read and archive__read")
    merged = [t for s, ts in listed.items() for t in mcp_to_loop_tools(ts, prefix=f"{s}__")] + [FINAL_ANSWER]
    assert len({t["name"] for t in merged}) == len(merged)
    per_tool = words(mcp_to_loop_tools(listed["docs"])) / len(listed["docs"])
    print(f"  each tool description is about {per_tool:.0f} words, re-sent on every model call:")
    for n_tools in [3, 30, 100, 300]:
        print(f"    {n_tools:3d} tools -> {n_tools * per_tool:6.0f} words per call, {6 * n_tools * per_tool:7.0f} words for a 6-call task")
    for c in clients.values():
        c.close()

### 运行演示 / Run the demo (`mcp_from_scratch.py`)

In [ ]:
main()

## `mcp_sdk.py`

```text
MCP with the official SDK, and MCP tools vs LangChain tools / 用官方 SDK 实现 MCP，以及 MCP 工具与 LangChain 工具的对比.

Run / 运行 (needs `pip install mcp==2.3.0 langgraph==1.2.19`):
    python docs/22-mcp-skills-sop/code/mcp_sdk.py

Covers / 包含:
    1. the same server in a few lines with the official Python SDK (MCPServer): the type hints and
       docstrings become the JSON schemas
       用官方 Python SDK（MCPServer）几行写出同一个服务器：类型注解和文档字符串变成 JSON schema
    2. interoperability: our hand-written client talks to the SDK server, and the SDK client talks to
       our hand-written server; that is the point of a protocol
       互通：手写的客户端连 SDK 服务器，SDK 客户端连手写的服务器；这正是“协议”的意义
    3. MCP tools inside LangChain / LangGraph: an adapter turns each MCP tool into a LangChain tool,
       and chapter 21's LangGraph agent runs on them unchanged
       在 LangChain / LangGraph 里用 MCP 工具：适配器把每个 MCP 工具变成 LangChain 工具，21 章的 LangGraph agent 原样运行
```

In [ ]:
__file__ = str(CODE / "mcp_sdk.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import asyncio
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

In [ ]:
HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent.parent / "21-agents" / "code"))

In [ ]:
from mcp_from_scratch import FINAL_ANSWER, StdioClient, calculate, mcp_to_loop_tools, make_caller, server_command  # noqa: E402
from agent_loop import ScriptedModel, agent_loop, make_tasks, make_world, run_tool  # noqa: E402

In [ ]:
try:
    from langchain_core.messages import HumanMessage
    from langchain_core.tools import StructuredTool
    from mcp import Client
    from mcp.client.stdio import StdioServerParameters
    from mcp.server import MCPServer
except ImportError:
    raise SystemExit("This file needs the MCP SDK and LangGraph: pip install mcp==2.3.0 langgraph==1.2.19 / 需要先安装")
WORLD = make_world()

### 1. The server with the SDK / 用 SDK 写服务器

In [ ]:
sdk_server = MCPServer("docs-sdk", instructions="Documents about made-up people and companies.")

In [ ]:
@sdk_server.tool()
def search(query: str) -> str:
    """Search the documents. Returns the titles of the best matches."""
    return run_tool(WORLD, "search", {"query": query})

In [ ]:
@sdk_server.tool()
def read(title: str) -> str:
    """Read the document with this exact title."""
    return run_tool(WORLD, "read", {"title": title})

In [ ]:
@sdk_server.tool()
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as 2026 - 1950."""
    return calculate(WORLD, expression)

In [ ]:
@sdk_server.resource("doc://{title}")
def document(title: str) -> str:
    """One document, by title."""
    return WORLD["docs"].get(title.replace("%20", " "), "")

In [ ]:
@sdk_server.prompt()
def answer_with_sources(question: str) -> str:
    """Answer a question from documents, citing them."""
    return f"Answer the question using only the documents and cite each title you used.\nQuestion: {question}"

### 3. MCP tools -> LangChain tools / MCP 工具 -> LangChain 工具

In [ ]:
def mcp_to_langchain(client, mcp_tools):
    """Wrap each MCP tool as a LangChain StructuredTool. The JSON schema from tools/list becomes the
    LangChain tool's schema; calling it sends tools/call to the server.
    把每个 MCP 工具包装成 LangChain 的 StructuredTool：tools/list 给出的 JSON schema 就是工具的 schema，调用时向服务器发 tools/call。"""
    def wrap(tool):
        def run(**kwargs):
            text, _ = client.call_tool(tool["name"], kwargs)
            return text
        return StructuredTool.from_function(func=run, name=tool["name"], description=tool["description"], args_schema=tool["inputSchema"])
    return [wrap(t) for t in mcp_tools]

In [ ]:
async def sdk_client_session(command):
    """The official client against any stdio server. 官方客户端连接任意 stdio 服务器。"""
    async with Client(StdioServerParameters(command=command[0], args=command[1:])) as client:
        tools = (await client.list_tools()).tools
        result = await client.call_tool("read", {"title": "Tafis Labs"})
        resource = await client.read_resource("doc://Tafis%20Labs")
        return [t.name for t in tools], result.content[0].text, resource.contents[0].text

In [ ]:
def run_async(coro):
    """asyncio.run, also inside Jupyter (whose event loop is already running). 在 Jupyter 里也能用的 asyncio.run。"""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(asyncio.run, coro).result()

In [ ]:
def main():
    print("== 1. the server with the SDK / 用 SDK 写的服务器 ==")
    ours = StdioClient(server_command(), name="hand-written-client")
    sdk = StdioClient(server_command(__file__), name="hand-written-client")
    ours_tools, sdk_tools = ours.list_tools(), sdk.list_tools()
    print(f"  SDK server: {sdk.discover()['_meta']['io.modelcontextprotocol/serverInfo']}")
    print(f"  schema generated from `def read(title: str) -> str`: {sdk_tools[1]['inputSchema']}")
    for a, b in zip(ours_tools, sdk_tools):
        assert (a["name"], a["description"], a["inputSchema"]["required"]) == (b["name"], b["description"], b["inputSchema"]["required"])

    print("\n== 2. interoperability / 互通 ==")
    text, _ = sdk.call_tool("read", {"title": "Tafis Labs"})
    print(f"  hand-written client -> SDK server: read -> {text}")
    names, text2, res = run_async(sdk_client_session(server_command()))
    print(f"  SDK client -> hand-written server: tools {names}; read -> {text2}; resources/read -> {res[:30]}...")
    assert text == text2 == res
    world_tasks = make_tasks(WORLD, n=20)
    same = total = 0
    for client in [ours, sdk]:
        tools = mcp_to_loop_tools(client.list_tools()) + [FINAL_ANSWER]
        call = make_caller({t["name"]: (client, t["name"]) for t in tools if t["name"] != "final_answer"})
        for k, ts in world_tasks.items():
            for question, truth in ts:
                answer = agent_loop(ScriptedModel(WORLD), WORLD, question, tools=tools, call_tool=call)[0]
                same += answer == truth
                total += 1
    print(f"  the chapter 21 loop on either server: {same}/{total} correct")
    assert same == total

    print("\n== 3. MCP tools in LangGraph / 在 LangGraph 中使用 MCP 工具 ==")
    from langgraph_agent import build_agent, tool_trace
    lc_tools = mcp_to_langchain(sdk, sdk_tools)
    print(f"  {len(lc_tools)} LangChain tools made from tools/list; schema of read: {lc_tools[1].args}")
    app = build_agent(ScriptedModel(WORLD), tools=lc_tools)
    same = total = 0
    for k, ts in world_tasks.items():
        for question, truth in ts:
            out = app.invoke({"messages": [HumanMessage(question)]})
            hand = agent_loop(ScriptedModel(WORLD), WORLD, question)
            same += out["messages"][-1].content == truth and tool_trace(out["messages"]) == tool_trace(hand[3])
            total += 1
    print(f"  {same}/{total} tasks: the LangGraph agent with MCP-backed tools makes the same tool calls as chapter 21")
    assert same == total
    print("  a LangChain tool is a Python object inside one program; an MCP tool lives in a separate server that any")
    print("  MCP host (any language, any framework) can discover and call. The adapter joins the two.")
    ours.close()
    sdk.close()

### 运行演示 / Run the demo (`mcp_sdk.py`)

In [ ]:
main()

## `skills.py`

```text
Agent Skills: instructions and scripts loaded on demand / Agent Skills：按需加载的说明和脚本.

Run / 运行:
    python docs/22-mcp-skills-sop/code/skills.py

Covers / 包含:
    1. a skill is a folder with a SKILL.md (YAML frontmatter: name, description; then instructions)
       and optional scripts/ and references/; parsing and validating it
       一个 skill 就是一个文件夹：SKILL.md（YAML 头：name、description，后面是说明），可选的 scripts/ 和 references/；解析与校验
    2. progressive disclosure: only names and descriptions sit in the prompt; the body is loaded when
       a task needs it; files and scripts are read or run only when a step says so
       渐进式披露：提示里只放名字和描述；任务需要时才加载正文；文件和脚本在某一步用到时才读取或运行
    3. an agent (chapter 21's loop) that loads a skill and follows its steps, and what it costs in context
       一个 agent（21 章的循环）加载 skill 并按步骤执行，以及上下文成本
    4. choosing a skill sees only the descriptions: good, terse and vague descriptions as the library grows
       选择 skill 时只看得到描述：随着 skill 越来越多，好的、简短的和含糊的描述各有多准

The "model" is again a script: it picks a skill by word overlap with the descriptions and follows the
numbered steps literally. A real model reads the same text as prose.
“模型”仍然是脚本：按与描述的词重叠挑选 skill，并逐字执行编号步骤。真实的模型把同样的文字当作普通说明来读。
```

In [ ]:
__file__ = str(CODE / "skills.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import json
import math
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

In [ ]:
HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent.parent / "21-agents" / "code"))

In [ ]:
from agent_loop import TOOLS as DOC_TOOLS  # noqa: E402
from agent_loop import agent_loop, make_tasks, make_world, run_tool  # noqa: E402

In [ ]:
SKILLS_DIR = HERE / "skills"
NAME_RE = re.compile(r"^[a-z0-9]+(-[a-z0-9]+)*$")

### 1. Reading and validating skills / 读取与校验

In [ ]:
def parse_skill(folder):
    """SKILL.md = '---', YAML frontmatter, '---', Markdown body. Handles the simple YAML skills use
    (key: value, and one level of nesting). SKILL.md = YAML 头 + Markdown 正文；这里只解析 skill 用到的简单 YAML。"""
    text = (folder / "SKILL.md").read_text(encoding="utf-8")
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.S)
    if not m:
        raise ValueError(f"{folder.name}: SKILL.md must start with YAML frontmatter between --- lines")
    meta, parent = {}, None
    for line in m.group(1).splitlines():
        key, _, value = line.strip().partition(":")
        value = value.strip().strip('"')
        if line.startswith("  ") and parent:
            meta[parent][key] = value
        elif value:
            meta[key], parent = value, None
        else:
            meta[key], parent = {}, key
    files = sorted(str(p.relative_to(folder)) for p in folder.rglob("*") if p.is_file() and p.name != "SKILL.md" and "__pycache__" not in p.parts)
    return {"name": meta.get("name", ""), "description": meta.get("description", ""), "meta": meta,
            "body": m.group(2).strip(), "dir": folder, "files": files}

In [ ]:
def validate(skill):
    """The Agent Skills spec's rules for the two required fields. Agent Skills 规范对两个必填字段的要求。"""
    name, desc, problems = skill["name"], skill["description"], []
    if not NAME_RE.match(name) or len(name) > 64:
        problems.append("name: 1-64 lowercase letters, digits and single hyphens")
    if name != skill["dir"].name:
        problems.append("name must match the folder name")
    if not 1 <= len(desc) <= 1024:
        problems.append("description: 1-1024 characters")
    return problems

In [ ]:
def load_library(root=SKILLS_DIR):
    skills = {}
    for folder in sorted(p for p in root.iterdir() if (p / "SKILL.md").exists()):
        skill = parse_skill(folder)
        problems = validate(skill)
        if problems:
            raise ValueError(f"{folder.name}: {problems}")
        skills[skill["name"]] = skill
    return skills

In [ ]:
def n_words(text):
    return len(text.split())

### 2. Progressive disclosure: the three levels as tools / 渐进式披露：三个层级对应三个工具

In [ ]:
def catalog(skills):
    """Level 1: always in the system prompt. 第 1 层：始终在系统提示里。"""
    return "Skills (call load_skill before using one):\n" + "\n".join(f"- {s['name']}: {s['description']}" for s in skills.values())

In [ ]:
SKILL_TOOLS = [
    {"name": "load_skill", "description": "Load the full instructions of a skill.",
     "parameters": {"type": "object", "properties": {"name": {"type": "string"}}, "required": ["name"]}},
    {"name": "read_skill_file", "description": "Read a file inside a skill folder, e.g. references/template.md.",
     "parameters": {"type": "object", "properties": {"name": {"type": "string"}, "path": {"type": "string"}}, "required": ["name", "path"]}},
    {"name": "run_skill_script", "description": "Run a script inside a skill folder with arguments separated by spaces.",
     "parameters": {"type": "object", "properties": {"name": {"type": "string"}, "path": {"type": "string"}, "args": {"type": "string"}}, "required": ["name", "path", "args"]}},
]

In [ ]:
def skill_file(skills, name, path):
    """Resolve a path inside the skill folder; refuse anything outside it (e.g. ../../secrets).
    解析 skill 文件夹里的路径；拒绝任何指向文件夹外面的路径。"""
    if name not in skills:
        return None
    root = skills[name]["dir"].resolve()
    target = (root / path).resolve()
    return target if target.is_relative_to(root) and target.is_file() else None

In [ ]:
def make_tool_runner(world, skills, log):
    def call(name, args):
        if name in ("search", "read", "calculator"):
            return run_tool(world, name, args)
        if name == "load_skill":  # level 2 / 第 2 层
            log.append(("body", args["name"]))
            return skills[args["name"]]["body"] if args["name"] in skills else f"No skill named {args['name']!r}."
        target = skill_file(skills, args["name"], args["path"])
        if target is None:
            return f"Error: no file {args['path']!r} in skill {args['name']!r}."
        log.append(("file", f"{args['name']}/{args['path']}"))  # level 3 / 第 3 层
        if name == "read_skill_file":
            return target.read_text(encoding="utf-8")
        out = subprocess.run([sys.executable, str(target), *args["args"].split()], capture_output=True, text=True, timeout=10)
        return (out.stdout or out.stderr).strip()  # the script's code never enters the context, only its output / 只有输出进入上下文
    return call

### 3. A scripted model that uses skills / 会用 skill 的脚本模型

In [ ]:
def tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

In [ ]:
class SkillFollower:
    """Picks the skill whose description shares the most words with the question, loads it, then
    executes its numbered steps that start with a `command`, filling <placeholders> from the question
    and from what it has read. 选出与问题词重叠最多的 skill，加载它，然后执行其中以 `命令` 开头的编号步骤。"""

    STOP = set("a an the of to and or is in for with what how many when use user asks it its from".split())

    def __init__(self, skills):
        self.skills = skills

    def pick(self, question):
        q = set(tokens(question)) - self.STOP
        return max(self.skills, key=lambda n: len(q & set(tokens(self.skills[n]["description"]))))

    def __call__(self, messages):
        question = messages[1]["content"]
        results = [m["content"] for m in messages if m["role"] == "tool"]
        loaded = next((r for r in results if r.startswith("#")), None)
        if loaded is None:
            return [{"name": "load_skill", "args": {"name": self.pick(question)}}]
        name = self.pick(question)
        steps = re.findall(r"^\d+\. `([^`]+)`", loaded, re.M)
        observed = " ".join(results)
        people = re.findall(r"(?:of|does|company|for whom) ([A-Z]\w+ [A-Z]\w+)", question) + re.findall(r"and ([A-Z]\w+ [A-Z]\w+)", question)
        companies = list(dict.fromkeys(re.search(rf"{p} works at ([A-Z]\w+ \w+)\.", observed).group(1) for p in people
                                       if re.search(rf"{p} works at ", observed)))
        years = {c: m.group(1) for c in companies if (m := re.search(rf"{c} was founded in (\d+)", observed))}
        done = len(results) - 1  # tool results after the skill body / 加载正文之后的工具结果数
        calls = []
        for step in steps:
            if step == "read <person>":
                calls += [{"name": "read", "args": {"title": p}} for p in people]
            elif step == "read <company>":
                calls += [{"name": "read", "args": {"title": c}} for c in companies or ["?"]]
            elif step.startswith("run "):
                script, *params = re.findall(r"<[^>]+>|\S+", step[4:])
                fill = {"<founding year>": 0, "<year 1>": 0, "<year 2>": -1}  # which company's year / 取哪家公司的年份
                values = [years.get(companies[fill[p]], "?") if companies and p in fill else p for p in params]
                calls.append({"name": "run_skill_script", "args": {"name": name, "path": script, "args": " ".join(values)}})
        if done < len(calls):
            return [calls[done]]
        if "references/template.md" in loaded and not any(r.startswith("# Profile template") for r in results):
            return [{"name": "read_skill_file", "args": {"name": name, "path": "references/template.md"}}]
        script_out = next(r for r in reversed(results) if not r.startswith("#"))  # the script's output / 脚本的输出
        if any(r.startswith("# Profile template") for r in results):
            template = next(r for r in results if r.startswith("# Profile template")).splitlines()[-1]
            city = re.search(rf"{companies[0]} was founded in \d+\. Its headquarters are in (\w+)", observed).group(1)
            answer = template.format(company=companies[0], city=city, year=years[companies[0]], age=script_out, person=people[0])
        else:
            answer = script_out
        return [{"name": "final_answer", "args": {"answer": answer}}]

### 4. Choosing among many skills: only the descriptions are visible / 在很多 skill 中选择：只看得到描述

In [ ]:
OBJECTS = {"csv": ["csv file", "spreadsheet", "table"], "pdf": ["pdf", "pdf document", "scanned form"],
           "slides": ["slide deck", "presentation", "slides"], "sql": ["sql query", "database query", "select statement"],
           "git": ["git commit", "commit history", "repository log"], "invoice": ["invoice", "bill", "receipt"],
           "email": ["email", "inbox message", "mail thread"], "test": ["unit test", "test suite", "pytest file"],
           "docker": ["dockerfile", "container image", "docker setup"], "log": ["server log", "log file", "error trace"],
           "contract": ["contract", "agreement", "terms of service"], "resume": ["resume", "cv", "job application"],
           "api": ["api spec", "openapi schema", "endpoint docs"], "image": ["image", "photo", "screenshot"],
           "chart": ["chart", "plot", "graph"], "notes": ["meeting notes", "minutes", "call transcript"]}
ACTIONS = {"summarize": ["summarize", "condense", "give the gist of"], "convert": ["convert", "export", "change the format of"],
           "validate": ["validate", "lint", "check for errors in"], "generate": ["generate", "draft", "create"],
           "translate": ["translate", "localize", "put into another language"], "clean": ["clean up", "tidy", "fix the formatting of"],
           "compare": ["compare", "diff", "find the differences between"], "extract": ["extract fields from", "parse", "pull data out of"]}

In [ ]:
def synthetic_library():
    lib = []
    for a, acts in ACTIONS.items():
        for o, objs in OBJECTS.items():
            lib.append({"name": f"{o}-{a}", "good": f"{acts[0].capitalize()} {objs[0]}s. Use when the user wants to {acts[1]} or {acts[2]} "
                                                    f"one; also called {objs[1]} or {objs[2]}.",
                        "terse": f"{acts[0].capitalize()} {objs[0]}s.", "vague": f"Helps with {objs[0]}s.", "acts": acts, "objs": objs})
    return lib

In [ ]:
def bm25_pick(question, descriptions, rng, k1=1.2, b=0.75):
    """A keyword matcher stands in for the model choosing a skill from its description; ties are broken at random.
    用关键词匹配代替模型根据描述挑选 skill；同分时随机选。"""
    docs = [Counter(tokens(d)) for d in descriptions]
    avg = sum(sum(d.values()) for d in docs) / len(docs)
    df = Counter(w for d in docs for w in d)
    q = set(tokens(question)) - SkillFollower.STOP
    scores = []
    for d in docs:
        length = sum(d.values())
        scores.append(sum(math.log(1 + (len(docs) - df[w] + 0.5) / (df[w] + 0.5)) * d[w] * (k1 + 1) / (d[w] + k1 * (1 - b + b * length / avg))
                          for w in q if w in d))
    best = max(scores)
    return rng.choice([i for i, s in enumerate(scores) if s == best])

In [ ]:
def selection_accuracy(style, size, n=400, seed=0):
    rng = random.Random(seed)
    lib = synthetic_library()
    hits = 0
    for _ in range(n):
        subset = rng.sample(lib, size)
        target = rng.randrange(size)
        s = subset[target]
        question = f"Can you {rng.choice(s['acts'])} this {rng.choice(s['objs'])} for me?"  # the user's own words / 用户自己的说法
        hits += bm25_pick(question, [x[style] for x in subset], rng) == target
    return hits / n

In [ ]:
def main():
    print("== 1. skills on disk / 磁盘上的 skill ==")
    skills = load_library()
    for s in skills.values():
        print(f"  {s['name']:16s} metadata {n_words(s['name'] + ' ' + s['description']):3d} words | body {n_words(s['body']):3d} words | files {s['files']}")
    bad = {"name": "Company_Profile", "description": "", "dir": Path("company-profile")}
    print(f"  a broken skill: {validate(bad)}")
    assert len(validate(bad)) == 3

    print("\n== 2-3. an agent that loads skills on demand / 按需加载 skill 的 agent ==")
    world = make_world()
    tasks = make_tasks(world, n=20)
    profile_tasks = [(f"Write a profile of the company {q.split('company ')[1].split(' works')[0]} works for.", None) for q, _ in tasks[2]]
    gap_tasks = tasks[5]
    tools = DOC_TOOLS[:3] + SKILL_TOOLS + DOC_TOOLS[3:]
    model = SkillFollower(skills)
    log = []
    question = profile_tasks[0][0]
    answer, steps, words_pd, messages = agent_loop(model, world, question, tools=tools, call_tool=make_tool_runner(world, skills, log))
    for m in messages[1:]:
        print(f"  {m['role']:9s} {m['content'][:120].replace(chr(10), ' ')}")
    print(f"  loaded: {log}")
    ok = 0
    for q, truth in profile_tasks + gap_tasks:
        a = agent_loop(SkillFollower(skills), world, q, tools=tools, call_tool=make_tool_runner(world, skills, []))[0]
        if truth is None:  # check the profile against the documents / 用文档核对简介
            person = q.split("company ")[1].split(" works")[0]
            company = re.search(r"works at (.+)\.$", world["docs"][person]).group(1)
            year = world["companies"][company]["founded"]
            truth = f"{company} is a company headquartered in {world['companies'][company]['city']}. It was founded in {year}, {2026 - year} years ago. {person} works there. Sources: [{person}], [{company}]."
        ok += a == truth
    print(f"  {ok}/{len(profile_tasks) + len(gap_tasks)} tasks answered correctly (profiles and year gaps)")
    assert ok == len(profile_tasks) + len(gap_tasks)

    all_bodies = "\n\n".join(f"## {s['name']}\n{s['body']}" for s in skills.values())
    per_call_pd = n_words(catalog(skills))
    per_call_inline = n_words(all_bodies)
    print(f"  in every model call: catalog {per_call_pd} words (progressive disclosure) vs {per_call_inline} words with every body inline;")
    print(f"  the one body this task needed ({n_words(skills['company-profile']['body'])} words) is read only after it is loaded")
    assert per_call_pd < per_call_inline

    print("\n== 4. choosing from descriptions only / 只看描述来选择 ==")
    lib = synthetic_library()
    print(f"  {len(lib)} synthetic skills, e.g. good: {lib[9]['good']!r}; terse: {lib[9]['terse']!r}; vague: {lib[9]['vague']!r}")
    table = {style: [selection_accuracy(style, k) for k in [8, 32, 128]] for style in ["good", "terse", "vague"]}
    for style, row in table.items():
        print(f"  {style:5s} descriptions, right skill picked with 8 / 32 / 128 skills: " + " / ".join(f"{v:.0%}" for v in row))
    assert table["good"][2] > table["terse"][2] + 0.2 > table["vague"][2] + 0.2
    assert table["vague"][0] > table["vague"][2]

### 运行演示 / Run the demo (`skills.py`)

In [ ]:
main()

## `sop.py`

```text
SOP-driven multi-agent collaboration / 由 SOP 驱动的多智能体协作.

Run / 运行:
    python docs/22-mcp-skills-sop/code/sop.py

Covers / 包含:
    1. an SOP (standard operating procedure) as data: each role's input, output schema and checks,
       wired together through a publish/subscribe message pool, as in MetaGPT
       把 SOP（标准作业程序）写成数据：每个角色的输入、输出 schema 和检查项，通过发布/订阅的消息池连接，与 MetaGPT 相同
    2. why handoffs need structure: agents that pass free-form text along a chain lose and garble
       facts at every step; structured artifacts with a schema check catch the missing ones
       为什么交接需要结构：沿着链条传递自由文本，每一步都会丢失和弄错事实；带 schema 校验的结构化产物能发现缺失
    3. a reviewer that checks values against the sources catches the garbled ones; what it all costs
       对照来源核对数值的审阅者能发现弄错的值；以及这一切的代价

Agents here are scripts that make mistakes at a set rate when they write (drop or alter a fact), so
the effect of the procedure is measurable. The documents are chapter 21's world, served by the MCP
server from mcp_from_scratch.py (in-process).
这里的 agent 是在“写”的时候按设定概率犯错（漏掉或改错一个事实）的脚本，这样流程的作用可以测量。
文档来自 21 章的世界，由 mcp_from_scratch.py 的 MCP 服务器提供（同一进程内）。
```

In [ ]:
__file__ = str(CODE / "sop.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import json
import random
import re
import sys
from pathlib import Path

In [ ]:
HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

In [ ]:
from mcp_from_scratch import META_KEY, PROTOCOL, DocsServer  # noqa: E402

In [ ]:
sys.path.insert(0, str(HERE.parent.parent / "21-agents" / "code"))

In [ ]:
from agent_loop import make_world  # noqa: E402

In [ ]:
FIELDS = ["person", "company", "city", "founded", "age"]
# The SOP: who does what, what they hand over, and what is checked / SOP：谁做什么、交出什么、检查什么
SOP = [
    {"role": "Researcher", "subscribes": "task", "publishes": "facts", "fields": ["person", "company", "city", "founded"],
     "how": "read the person's document, then the company's"},
    {"role": "Analyst", "subscribes": "facts", "publishes": "analysis", "fields": FIELDS,
     "how": "age = 2026 - founded, with the calculator"},
    {"role": "Writer", "subscribes": "analysis", "publishes": "draft", "fields": FIELDS, "how": "write the report"},
    {"role": "Editor", "subscribes": "draft", "publishes": "edited", "fields": FIELDS, "how": "tighten the wording"},
    {"role": "Translator", "subscribes": "edited", "publishes": "translated", "fields": FIELDS, "how": "localize the report"},
    {"role": "Publisher", "subscribes": "translated", "publishes": "report", "fields": FIELDS, "how": "format for publication"},
]

In [ ]:
class Tools:
    """The agents' tools: JSON-RPC calls to the documents' MCP server. agent 的工具：对文档 MCP 服务器的 JSON-RPC 调用。"""

    def __init__(self, world):
        self.server, self.calls = DocsServer(world), 0

    def call(self, name, **arguments):
        self.calls += 1
        msg = {"jsonrpc": "2.0", "id": self.calls, "method": "tools/call",
               "params": {"name": name, "arguments": arguments, "_meta": {META_KEY + "protocolVersion": PROTOCOL, META_KEY + "clientCapabilities": {}}}}
        return self.server.handle(msg)["result"]["content"][0]["text"]

In [ ]:
class Pool:
    """MetaGPT's shared message pool: roles publish typed messages and subscribe to the types they need.
    MetaGPT 的共享消息池：角色发布带类型的消息，并订阅自己需要的类型。"""

    def __init__(self):
        self.messages = []

    def publish(self, kind, sender, content):
        self.messages.append({"kind": kind, "from": sender, "content": content})

    def latest(self, kind):
        return next(m["content"] for m in reversed(self.messages) if m["kind"] == kind)

In [ ]:
class Writer:
    """How an imperfect agent writes: each fact may be dropped or altered. 不完美的 agent 怎样“写”：每个事实都可能被漏掉或改错。"""

    def __init__(self, world, p_drop, p_alter, rng):
        self.world, self.p_drop, self.p_alter, self.rng = world, p_drop, p_alter, rng
        self.cities = sorted({c["city"] for c in world["companies"].values()})

    def alter(self, field, value):
        if field in ("founded", "age"):
            return str(int(value) + self.rng.choice([-10, -1, 1, 10]))
        pool = {"company": sorted(self.world["companies"]), "city": self.cities, "person": sorted(self.world["people"])}[field]
        return self.rng.choice([v for v in pool if v != value])

    def write(self, facts):
        out = {}
        for f, v in facts.items():
            r = self.rng.random()
            if r < self.p_drop:
                continue
            out[f] = self.alter(f, v) if r < self.p_drop + self.p_alter else v
        return out

In [ ]:
def to_prose(facts):
    """Free-form handoff: the facts as a sentence (missing facts simply aren't mentioned). 自由文本交接。"""
    parts = []
    if "person" in facts or "company" in facts:
        parts.append(f"{facts.get('person', 'Someone')} works at {facts.get('company', 'a company')}")
    if "city" in facts:
        parts.append(f"headquartered in {facts['city']}")
    if "founded" in facts:
        parts.append(f"founded in {facts['founded']}")
    if "age" in facts:
        parts.append(f"{facts['age']} years old")
    return ", ".join(parts) + "."

In [ ]:
def from_prose(text):
    """The next agent reads the sentence back. 下一个 agent 把句子读回来。"""
    patterns = {"person": r"^([A-Z]\w+ [A-Z]\w+) works at", "company": r"works at ([A-Z]\w+ [A-Z]\w+)", "city": r"headquartered in (\w+)",
                "founded": r"founded in (\d+)", "age": r"(\d+) years old"}
    return {f: m.group(1) for f, p in patterns.items() if (m := re.search(p, text))}

In [ ]:
def research(tools, person):
    """The first role's real work: two tool calls. 第一个角色的实际工作：两次工具调用。"""
    doc = tools.call("read", title=person)
    company = re.search(r"works at (.+)\.$", doc).group(1)
    doc2 = tools.call("read", title=company)
    founded, city = re.search(r"founded in (\d+)\. Its headquarters are in (\w+)", doc2).groups()
    return {"person": person, "company": company, "city": city, "founded": founded}

In [ ]:
def verify(tools, facts, task):
    """Reviewer's checklist: starting from the original task (not the last message), re-read the sources
    and recompute; return the corrected facts and how many were wrong.
    审阅者的检查清单：从最初的任务（而不是上一条消息）出发，重读来源、重新计算；返回改正后的事实和改错了几个。"""
    true = research(tools, task["person"])
    true["age"] = tools.call("calculator", expression=f"2026 - {true['founded']}").split(" = ")[1]
    wrong = sum(facts.get(f) != true[f] for f in FIELDS)
    return true, wrong

In [ ]:
def run(world, person, stages, mode, p_drop, p_alter, rng, max_redo=3):
    """One task through the first `stages` roles of the SOP. mode: 'free' (prose handoffs), 'schema'
    (JSON artifacts, missing fields sent back), 'schema+review' (plus a reviewer checking the sources).
    一个任务经过 SOP 的前 stages 个角色。"""
    tools, pool, writer = Tools(world), Pool(), Writer(world, p_drop, p_alter, rng)
    pool.publish("task", "user", {"person": person})
    messages = 1
    for i, role in enumerate(SOP[:stages]):
        if i == 0:
            facts = research(tools, person)
        else:
            facts = pool.latest(role["subscribes"])
            facts = from_prose(facts) if mode == "free" else dict(facts)
            if role["role"] == "Analyst" and "founded" in facts:
                facts["age"] = tools.call("calculator", expression=f"2026 - {facts['founded']}").split(" = ")[1]
        expected = role["fields"] if stages > 1 else ["person", "company", "city", "founded"]
        out = writer.write(facts)
        if mode != "free":  # the receiver checks the artifact against the schema / 接收方按 schema 检查
            for _ in range(max_redo):
                missing = [f for f in expected if f in facts and f not in out]
                if not missing:
                    break
                messages += 2  # "missing X" back, a corrected artifact forward / 退回“缺少 X”，再收到改正后的产物
                out |= {f: v for f, v in writer.write({f: facts[f] for f in missing}).items()}
        pool.publish(role["publishes"], role["role"], to_prose(out) if mode == "free" else out)
        messages += 1
    final = pool.latest(SOP[stages - 1]["publishes"])
    final = from_prose(final) if mode == "free" else final
    if mode == "schema+review":
        final, _ = verify(tools, final, pool.latest("task"))
        messages += 1
    truth = research(Tools(world), person)
    truth["age"] = str(2026 - int(truth["founded"]))
    needed = FIELDS if stages > 1 else ["person", "company", "city", "founded"]
    ok = all(final.get(f) == truth[f] for f in needed)
    return ok, messages, tools.calls

In [ ]:
def evaluate(world, stages, mode, p_drop=0.02, p_alter=0.02, n=300, seed=0):
    rng = random.Random(seed)
    people = sorted(world["people"])
    res = [run(world, rng.choice(people), stages, mode, p_drop, p_alter, rng) for _ in range(n)]
    return sum(r[0] for r in res) / n, sum(r[1] for r in res) / n, sum(r[2] for r in res) / n

In [ ]:
def main():
    world = make_world()
    print("== 1. the SOP / SOP ==")
    for r in SOP:
        print(f"  {r['role']:10s} reads {r['subscribes']:10s} -> publishes {r['publishes']:10s} fields {r['fields']}  ({r['how']})")
    rng = random.Random(4)
    print("\n  one free-form run through 6 roles, at 5% drop + 5% alter per fact per handoff:")
    tools, writer = Tools(world), Writer(world, 0.05, 0.05, rng)
    facts = research(tools, "Nolen Dadasel")
    facts["age"] = str(2026 - int(facts["founded"]))
    for r in SOP[1:]:
        facts = from_prose(to_prose(writer.write(facts)))
        print(f"    {r['role']:10s} says: {to_prose(facts)}")
    print("  the same handoff as an artifact checked against its schema:")
    print("    " + json.dumps({"kind": "analysis", "from": "Analyst", "content": {"person": "Nolen Dadasel", "company": "Tafis Labs", "city": "Begalberg", "founded": "1941", "age": "85"}}))

    print("\n== 2-3. success over a chain of roles, 2% drop + 2% alter per fact per handoff ==")
    table = {}
    for mode in ["free", "schema", "schema+review"]:
        table[mode] = {k: evaluate(world, k, mode) for k in [2, 4, 6]}
        print(f"  {mode:14s} " + " | ".join(f"{k} roles: {v[0]:4.0%} ({v[1]:.1f} msgs, {v[2]:.1f} tool calls)" for k, v in table[mode].items()))
    assert table["free"][2][0] > table["free"][6][0] + 0.1               # free-form decays with the chain / 自由文本随链条变差
    assert table["schema"][6][0] > table["free"][6][0] + 0.1             # schemas catch the drops / schema 能发现遗漏
    assert table["schema+review"][6][0] > table["schema"][6][0] + 0.1    # review catches the alterations / 审阅能发现改错
    assert table["schema+review"][6][2] > table["free"][6][2]             # and it costs tool calls / 代价是更多工具调用
    print("  a schema check catches what is missing but not what is wrong; checking against the sources catches both")

### 运行演示 / Run the demo (`sop.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.